In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q "datasets<4.0" jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [9]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [10]:
import os
import pandas as pd
from datasets import load_dataset, Audio

url = "https://raw.githubusercontent.com/Chichay317/nigerian-accented-clinical-asr/main/data/test_subset.csv"
test_df = pd.read_csv(url)

def key(p):
    return os.path.basename(str(p)).rsplit(".", 1)[0]

keep = set(test_df["audio_paths"].map(key))

rows = []
for acc in ["yoruba", "igbo", "hausa"]:
    ds = load_dataset("intronhealth/afrispeech-200", acc, split="test",
                      streaming=True, trust_remote_code=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    for ex in ds:
        if key(ex["path"]) in keep:
            rows.append({"accent": acc, "domain": ex["domain"],
                         "reference": ex["transcript"],
                         "audio": ex["audio"]["array"]})
    print(acc, sum(r["accent"] == acc for r in rows))

print("Total clips loaded:", len(rows))

README.md: 0.00B [00:00, ?B/s]

afrispeech-200.py: 0.00B [00:00, ?B/s]

accent_stats.py: 0.00B [00:00, ?B/s]

Reading metadata...: 648it [00:00, 1227.31it/s]


yoruba 642


Reading metadata...: 355it [00:00, 508.45it/s]


igbo 354


Reading metadata...: 196it [00:00, 372.54it/s]


hausa 189
Total clips loaded: 1185


In [12]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(
    model_id, dtype=torch.float16).to("cuda")
model.eval()

batch_size = 16
for i in range(0, len(rows), batch_size):
    batch = rows[i:i + batch_size]
    feats = processor.feature_extractor(
        [r["audio"] for r in batch], sampling_rate=16000,
        return_tensors="pt").input_features.to("cuda", dtype=torch.float16)
    with torch.no_grad():
        ids = model.generate(feats, language="english", task="transcribe",
                             max_new_tokens=225)
    for r, t in zip(batch, processor.batch_decode(ids, skip_special_tokens=True)):
        r["prediction"] = t
    if (i // batch_size) % 10 == 0:
        print(f"{i + len(batch)}/{len(rows)} clips done")

print("Done:", sum("prediction" in r for r in rows), "clips transcribed")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans

16/1185 clips done


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


176/1185 clips done


...

976/1185 clips done


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1136/1185 clips done


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done: 1185 clips transcribed


In [13]:
import jiwer
import pandas as pd

norm = processor.tokenizer.normalize

res = pd.DataFrame([{k: v for k, v in r.items() if k != "audio"} for r in rows])
res["ref_norm"] = res["reference"].map(norm)
res["pred_norm"] = res["prediction"].map(norm)
res = res[res["ref_norm"].str.strip() != ""]

overall = jiwer.wer(list(res["ref_norm"]), list(res["pred_norm"]))
print(f"Overall WER: {overall:.1%}\n")

by_group = res.groupby(["accent", "domain"]).apply(
    lambda g: jiwer.wer(list(g["ref_norm"]), list(g["pred_norm"])))
print((by_group * 100).round(1).rename("WER %"))

print("\nExamples:")
print(res[["reference", "prediction"]].head(5).to_string())

res.to_csv("baseline_predictions.csv", index=False)

Overall WER: 37.7%

accent  domain  
hausa   clinical    43.9
        general     26.3
igbo    clinical    38.9
        general     34.7
yoruba  clinical    43.8
        general     35.0
Name: WER %, dtype: float64

Examples:
                                                                                                                               reference                                                                                                                                      prediction
0                                                   A second impact tore open the second of the two watertight compartments flooding it.                                               A second impact, tall open the second of the two watertight compartments, flooding it, full stop.
1  X-ray examination of the mastoids is of little value though it may show areas of bone destruction and enlargement of jugular foramen.   X iPhone Re examination of the mastoid is of little value, though it may 

/tmp/ipykernel_58/1211049156.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_group = res.groupby(["accent", "domain"]).apply(
